# Calcinosis scRNA-seq Pipeline — Google Colab (GPU)

**Repository:** https://github.com/glenritschel/calcinosis-crest  
**Dataset:** GSE138669 (Tabib 2021, SSc skin biopsies, 22 samples)  
**Runtime required:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

## Before you start
1. Click **Runtime → Change runtime type → T4 GPU** (free tier)
2. Run all cells top-to-bottom — estimated time ~25 min on T4
3. Download results at the end using the Files panel (left sidebar)

---

## Cell 1 — Check GPU

In [ ]:
# Verify GPU is available — if this says False, go to Runtime → Change runtime type → T4 GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU, then re-run.')

## Cell 2 — Install dependencies

Takes ~5 minutes on first run. Colab caches between sessions on the same runtime.

In [ ]:
%%capture
# Pin versions to match local ssc-scvi environment
!pip install -q \
    'scvi-tools==1.3.3' \
    'scanpy==1.11.5' \
    'anndata==0.11.4' \
    'gseapy==1.1.10' \
    'mygene==3.2.2' \
    'GEOparse==2.0.4' \
    'scikit-misc' \
    'leidenalg' \
    'python-igraph'

print('Installation complete.')

In [ ]:
# Verify key versions
import scanpy, scvi, anndata, gseapy
print(f'scanpy   {scanpy.__version__}')
print(f'scvi     {scvi.__version__}')
print(f'anndata  {anndata.__version__}')
print(f'gseapy   {gseapy.__version__}')

## Cell 3 — Clone repo and set up directories

In [ ]:
import os

REPO = '/content/calcinosis-crest'

if not os.path.exists(REPO):
    !git clone https://github.com/glenritschel/calcinosis-crest {REPO}
else:
    !cd {REPO} && git pull

os.chdir(REPO)
print('Working directory:', os.getcwd())

# Create required directories
for d in ['data/raw/GSE138669', 'data/processed', 'data/processed/per_sample',
           'results/tables', 'results/drug_repurposing', 'figures']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

## Cell 4 — Download GSE138669

Downloads ~3.5 GB from NCBI FTP. Takes ~8 min on Colab's network.

In [ ]:
import os
from pathlib import Path

RAW = Path('data/raw/GSE138669')
h5_files = list(RAW.glob('*.h5'))

if len(h5_files) >= 10:  # expect 22 samples
    print(f'Data already present: {len(h5_files)} .h5 files found, skipping download.')
else:
    print('Downloading GSE138669 from NCBI FTP...')
    !python src/data_download.py
    h5_files = list(RAW.glob('*.h5'))
    print(f'Download complete: {len(h5_files)} .h5 files')

## Cell 5 — Preprocess

QC filtering, Ensembl→symbol mapping, HVG selection. Saves `calcinosis_qc.h5ad`.

In [ ]:
from pathlib import Path

qc_h5ad = Path('data/processed/calcinosis_qc.h5ad')

if qc_h5ad.exists():
    print(f'QC h5ad already exists ({qc_h5ad.stat().st_size / 1e6:.0f} MB), skipping preprocess.')
else:
    print('Running preprocess.py ...')
    !python src/preprocess.py
    print('Preprocess complete.')

## Cell 6 — scVI training (400 epochs, GPU)

This is the main compute step. ~8 minutes on T4 GPU vs ~90 minutes on CPU.  
Saves `calcinosis_scvi.h5ad` and `calcinosis_scvi_annot.h5ad`.

In [ ]:
from pathlib import Path

scvi_h5ad = Path('data/processed/calcinosis_scvi_annot.h5ad')

if scvi_h5ad.exists():
    print(f'scVI h5ad already exists, skipping training.')
    print(f'  Delete {scvi_h5ad} to retrain.')
else:
    print('Running modeling.py with 400 epochs on GPU...')
    !python src/modeling.py --epochs 400
    print('modeling.py complete.')

In [ ]:
# Verify outputs
import scanpy as sc
from pathlib import Path

annot = Path('data/processed/calcinosis_scvi_annot.h5ad')
assert annot.exists(), f'Missing {annot} — check modeling.py output above'

adata = sc.read_h5ad(annot)
print(f'Cells: {adata.n_obs:,}   Genes: {adata.n_vars:,}')
print(f'Obs columns: {list(adata.obs.columns)}')
print(f'Leiden clusters: {adata.obs["leiden"].nunique()}')

# Show calcinosis signatures if present
sig_cols = [c for c in adata.obs.columns if 'sig' in c.lower() or 'score' in c.lower()]
if sig_cols:
    print(f'Signature scores: {sig_cols}')

## Cell 7 — UMAP visualisation

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.figdir = 'figures/'
sc.settings.set_figure_params(dpi=100, figsize=(8, 6))

# Leiden clusters
sc.pl.umap(adata, color='leiden', legend_loc='on data',
           title='Leiden clusters (scVI, 400 epochs)', save='_leiden.png')

# Sample distribution
sc.pl.umap(adata, color='sample', title='Sample distribution', save='_sample.png')

# Calcinosis signatures
sig_cols = [c for c in adata.obs.columns if 'sig' in c.lower()]
if sig_cols:
    sc.pl.umap(adata, color=sig_cols[:4], ncols=2,
               title=[c.replace('_sig','').replace('_',' ').title() for c in sig_cols[:4]],
               save='_calcinosis_sigs.png')
    print(f'Signature UMAPs saved.')

## Cell 8 — Differential expression + LINCS drug candidates

In [ ]:
from pathlib import Path

de_csv = Path('results/tables/de_leiden_wilcoxon.csv')
lincs_csv = Path('results/drug_repurposing/lincs_candidates.csv')

if lincs_csv.exists():
    print(f'LINCS results already exist ({lincs_csv.stat().st_size / 1e3:.0f} KB), skipping DE.')
else:
    print('Running de_analysis.py ...')
    !python src/de_analysis.py
    print('DE analysis complete.')

In [ ]:
# Preview LINCS results
import pandas as pd
from pathlib import Path

lincs_csv = Path('results/drug_repurposing/lincs_candidates.csv')
if lincs_csv.exists():
    df = pd.read_csv(lincs_csv)
    print(f'LINCS candidates: {len(df):,} rows')
    print(f'Columns: {list(df.columns)}')
    # Show top 20 by score
    score_col = next((c for c in ['score','rev_score_sum','combined_score'] if c in df.columns), df.columns[-1])
    print(f'\nTop 20 by {score_col}:')
    display(df.nlargest(20, score_col)[df.columns[:6]])
else:
    print('LINCS CSV not found — check de_analysis.py output above.')

## Cell 9 — Download results

Zips everything and downloads to your machine. Then commit to GitHub.

In [ ]:
import os, shutil
from google.colab import files

# Zip results/ and figures/
print('Zipping results...')
shutil.make_archive('/content/calcinosis_results', 'zip', '.', 'results')
shutil.make_archive('/content/calcinosis_figures', 'zip', '.', 'figures')

print('Downloading results.zip ...')
files.download('/content/calcinosis_results.zip')

print('Downloading figures.zip ...')
files.download('/content/calcinosis_figures.zip')

In [ ]:
# Optional: also download the h5ad files (large — ~1-2 GB each)
# Uncomment if you want them locally

# from google.colab import files
# files.download('data/processed/calcinosis_scvi_annot.h5ad')
# files.download('data/processed/calcinosis_scvi.h5ad')

## Cell 10 — Commit results to GitHub (optional)

Requires a GitHub Personal Access Token with `repo` scope.  
Create one at: https://github.com/settings/tokens

In [ ]:
# Fill in your token and run this cell to push results directly from Colab
import os

GITHUB_TOKEN = ''   # paste your token here — do NOT commit this cell with the token
GITHUB_USER  = 'glenritschel'
REPO_NAME    = 'calcinosis-crest'

if GITHUB_TOKEN:
    remote = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    !cd /content/calcinosis-crest && \
        git config user.email 'glen@ritschel.com' && \
        git config user.name 'Glen Ritschel' && \
        git add results/ figures/ data/processed/calcinosis_scvi_annot.h5ad && \
        git commit -m 'Add 400-epoch scVI results (Colab T4 GPU run)' && \
        git push {remote} HEAD:main
    print('Pushed to GitHub.')
else:
    print('No token provided — download results manually from Cell 9.')

---
## Notes

**If modeling.py fails with 'No GPU backend':** The runtime lost its GPU. Go to Runtime → Disconnect and delete runtime → Reconnect → Change runtime type → T4 GPU → Re-run from Cell 6 (preprocessing is cached).

**If data download times out:** Run `!python src/data_download.py` again — it resumes from where it left off.

**Session timeout:** Colab free tier disconnects after ~90 min of inactivity. If this happens during training, re-run from Cell 6 — the h5ad files from preprocessing are still cached.

**Runtime:** ~25 min total on T4. Breakdown:
- Install: 5 min
- Download: 8 min  
- Preprocess: 4 min
- scVI 400 epochs: 8 min
- DE + LINCS: 5 min

**Authors:** Glen Ritschel & Claude (Anthropic), 2026